In [1]:
%load_ext autoreload
%autoreload 2

from functools import partial
from typing import Optional

from flax import nnx
from jaxtyping import Array
import tensorflow_datasets as tfds

import jax
import jax.numpy as jnp
import optax

from trainax._dataloader import JaxLoader
from trainax._trainer import NNXTrainer
from trainax._types import StepOutput, ValStepOutput

In [2]:
class CNN(nnx.Module):
  """A simple CNN model."""

  def __init__(self, *, rngs: nnx.Rngs):
    self.rngs = rngs
    self.conv1 = nnx.Conv(1, 32, kernel_size=(3, 3), rngs=rngs)
    self.batch_norm1 = nnx.BatchNorm(32, rngs=rngs)
    self.dropout1 = nnx.Dropout(rate=0.025, rngs=rngs)
    self.conv2 = nnx.Conv(32, 64, kernel_size=(3, 3), rngs=rngs)
    self.batch_norm2 = nnx.BatchNorm(64, rngs=rngs)
    self.avg_pool = partial(nnx.avg_pool, window_shape=(2, 2), strides=(2, 2))
    self.linear1 = nnx.Linear(3136, 256, rngs=rngs)
    self.dropout2 = nnx.Dropout(rate=0.025, rngs=rngs)
    self.linear2 = nnx.Linear(256, 10, rngs=rngs)

  def __call__(self, x):
    x = self.avg_pool(nnx.relu(self.batch_norm1(self.dropout1(self.conv1(x), rngs=self.rngs))))
    x = self.avg_pool(nnx.relu(self.batch_norm2(self.conv2(x))))
    x = x.reshape(x.shape[0], -1)  # flatten
    x = nnx.relu(self.dropout2(self.linear1(x), rngs=self.rngs))
    x = self.linear2(x)
    return x

# Instantiate the model.
model = CNN(rngs=nnx.Rngs(0))
# Visualize it.
nnx.display(model)

In [3]:
def loss_fn(model: CNN, data: dict[str, Array]):
    logits = model(data["x"])
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=data["label"]
    ).mean()
    return loss, logits


# @nnx.jit
def train_step(model: CNN, batch: dict[str, Array]):
    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(
        model, batch
    )
    return StepOutput(
        loss=loss,
        y=batch["label"],
        yhat=jax.nn.softmax(logits),
        gradients=grads,
    )


# @nnx.jit
def eval_step(model: CNN, batch: dict[str, Array], metrics: nnx.metrics.Metric | None = None):
    loss, logits = loss_fn(model, batch)

    preds = jax.nn.softmax(logits)
    acc = jnp.mean(jnp.argmax(preds) == batch["x"])

    if metrics is not None:
        metrics.update(loss=loss, logits=logits, labels=batch["label"])
    
    return ValStepOutput(
        loss,
        batch["x"],
        preds,
        metrics={"acc": acc}
    )

In [4]:
def get_datasets(batch_size: int = 1000, **kwargs) -> tuple[JaxLoader, JaxLoader]:
    """Load MNIST train and test datasets into memory."""
    ds_builder = tfds.builder("mnist")
    ds_builder.download_and_prepare()
    
    train_ds = tfds.as_numpy(ds_builder.as_dataset(split="train", batch_size=-1))
    test_ds = tfds.as_numpy(ds_builder.as_dataset(split="test", batch_size=-1))
    
    train_ds["x"] = jnp.float32(train_ds.pop("image")) / 255.0
    test_ds["x"] = jnp.float32(test_ds.pop("image")) / 255.0
    return JaxLoader(train_ds, batch_size, **kwargs), JaxLoader(test_ds, batch_size, **kwargs)

In [5]:
train_mnist, test_mnist = get_datasets()

In [6]:
train_mnist, test_mnist

(JaxLoader(Data attributes: label, x, Batch size: 1000, N batches: 60, Sharding: No sharding),
 JaxLoader(Data attributes: label, x, Batch size: 1000, N batches: 10, Sharding: No sharding))

In [7]:
from functools import partial
from trainax import NNXMetricTracker

In [ ]:
jax.config.update("jax_disable_jit", False)

learning_rate = 0.005
momentum = 0.9

optimizer = optax.adamw(learning_rate, momentum)
metrics = nnx.MultiMetric(
  accuracy=nnx.metrics.Accuracy(),
  loss=nnx.metrics.Average('loss'),
)

trainer = NNXTrainer(
    n_epochs=20,
    callbacks=[NNXMetricTracker(metrics)],
    # callbacks=[],
    val_every=5,
    use_rich=True,
)

trained_model, trained_state = trainer.train(
    model=model,
    optim=optimizer,
    train_step=train_step,
    trainloader=train_mnist,
    val_step=eval_step,
    valloader=test_mnist,
)

In [ ]:
trained_model